# H-GNN-RNN-DTI - Objective 1 Fusion Ablation (GPU)

Runs the core RQ1 experiment: **GCA vs concat vs flat_gate vs ungated_xattn** on DAVIS (S1 warm-start), 3 seeds each, reporting AUROC/AUPR mean±std + a paired significance test.

The **preprocessed DAVIS dataset is bundled in the repo**, so there is nothing to download and no RDKit / PyTDC to install - only `torch-geometric` on top of Colab's stock PyTorch. That keeps 'Run all' reliable.

**Before running:** `Runtime -> Change runtime type -> Hardware accelerator = GPU (T4)`, then `Runtime -> Run all`.

If the `git clone` cell fails, the code hasn't been pushed to the branch yet - push `objective1-impl` to `https://github.com/jamcy98/Modeling-DTI.git`, or upload the `H-GNN-RNN-DTI/` folder and `%cd` into it.


## 1. Check the GPU


In [ ]:
!nvidia-smi -L
import torch; print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())


## 2. Get the code (+ bundled preprocessed data)


In [ ]:
REPO_URL = 'https://github.com/jamcy98/Modeling-DTI.git'
BRANCH = 'objective1-impl'
!git clone --depth 1 --branch $BRANCH $REPO_URL
%cd Modeling-DTI/H-GNN-RNN-DTI
# sanity: the bundled processed DAVIS data should be present
import os; assert os.path.exists('data/processed/davis/pairs.csv'), 'processed DAVIS missing'
print('bundled DAVIS data found')


## 3. Install dependencies
Colab already ships a CUDA build of PyTorch, plus numpy / pandas / scipy / scikit-learn / pyyaml. We only need PyTorch Geometric. (No RDKit or PyTDC: the preprocessed graphs are already in the repo, and the training/ablation code imports RDKit lazily.)


In [ ]:
!pip -q install torch-geometric
print('deps installed')


## 4. Run the fusion ablation on GPU
Full DAVIS, all four fusions, seeds 42/43/44. On a T4 this is the main compute; expect roughly tens of minutes. For a fast dry run first, add `--epochs 2 --max-batches 30`.


In [ ]:
!python scripts/run_ablation.py --dataset davis --split S1 --device cuda --seeds 42 43 44


## 5. Show + download results


In [ ]:
import json
res = json.load(open('experiments/results/ablation_davis_S1.json'))
print(json.dumps(res['summary'], indent=2))
print(json.dumps(res['significance_gca_vs_baselines'], indent=2))


In [ ]:
from google.colab import files
files.download('experiments/results/ablation_davis_S1.json')


## What the result means
- **AUPR** is the honest metric here (DAVIS is ~4% positive).
- RQ1 is supported if **gca** beats concat / flat_gate / ungated_xattn on AUPR, and the paired test `dAUPR` is positive with a small p-value.
- If GCA does *not* beat the simpler fusions, that is a real (publishable) finding too - see docs/objective1_analysis.md Risk R1.

To regenerate the preprocessed data from scratch instead of using the bundled copy (needs RDKit + PyTDC): `pip install rdkit 'PyTDC==0.4.1' 'setuptools<81'` then `python scripts/download_data.py --dataset davis && python scripts/preprocess.py --dataset davis`.
